# ENT Triage Model Finetuning

Finetune a small LLM (Qwen2) for ENT triage using synthetic transcript→output pairs.

**Output format:** SUMMARY, FINDINGS, FLAGS, URGENCY (routine, semi-urgent, or urgent), REASONING

**Deployment note:** Groq does *not* host custom finetuned models. After finetuning, deploy to **Together.ai**, **Replicate**, or **Ollama** (local).

In [ ]:
# Install dependencies (run once)
!pip install -q transformers peft datasets accelerate bitsandbytes

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
import torch

## 1. Load or upload training data

Option A: Upload `triage_training_data.jsonl` to Colab.

Option B: Generate inline (small subset for quick test).

In [ ]:
# Option A: Load from uploaded file
# !pip install -q -U google-colab
# from google.colab import files
# uploaded = files.upload()  # Choose triage_training_data.jsonl
# DATA_PATH = list(uploaded.keys())[0]

# Option B: Use sample inline
SAMPLE_DATA = [
    {
        "transcript": "chief_complaint:mild sore throat. symptom_duration:3 days. symptom_severity:mild. symptom_progression:improving. red_flags:No. risk_factors:No",
        "output": "SUMMARY: Patient presents with mild sore throat of 3 days duration. Symptoms improving. No red flags.\nFINDINGS:\n- Mild sore throat\n- 3-day duration\n- Improving trend\nFLAGS: [SYMPTOM] sore throat, [SEVERITY] mild, [DURATION] 3 days, [PROGRESSION] improving\nURGENCY: routine\nREASONING: Mild symptoms, improving, no red flags. Routine appointment."
    },
    {
        "transcript": "chief_complaint:difficulty breathing. symptom_duration:few hours. symptom_severity:severe. symptom_progression:rapidly worsening. red_flags:Yes. risk_factors:No",
        "output": "SUMMARY: Patient with acute difficulty breathing, rapidly worsening. Critical red flag.\nFINDINGS:\n- Difficulty breathing\n- Rapid deterioration\nFLAGS: [RED_FLAG] difficulty breathing, [SEVERITY] severe\nURGENCY: urgent\nREASONING: Difficulty breathing is a critical red flag. Same-day evaluation required."
    },
]

def load_jsonl(path: str):
    data = []
    with open(path) as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

try:
    DATA_PATH = 'triage_training_data.jsonl'
    data = load_jsonl(DATA_PATH)
    print(f"Loaded {len(data)} examples from {DATA_PATH}")
except FileNotFoundError:
    data = SAMPLE_DATA
    print(f"Using {len(data)} sample examples (upload triage_training_data.jsonl for full training)")

# Limit for quick run
MAX_EXAMPLES = 500
data = data[:MAX_EXAMPLES]

In [ ]:
# Convert to chat format for instruction tuning
def to_chat(r):
    return {
        "messages": [
            {"role": "user", "content": f"Analyze this ENT triage transcript and produce the structured output. URGENCY must be one of: routine, semi-urgent, or urgent.\n\nTRANSCRIPT:\n{r['transcript']}"},
            {"role": "assistant", "content": r["output"]},
        ]
    }

chat_data = [to_chat(r) for r in data]
dataset = Dataset.from_list(chat_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"  # Small for Colab; use Qwen2-1.5B-Instruct for better quality

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
        return_tensors=None,
    )

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
train_ds = train_ds.with_format("torch")

training_args = TrainingArguments(
    output_dir="./triage_qwen_lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)
trainer.train()

In [ ]:
# Save LoRA adapter
trainer.save_model("./triage_qwen_lora")
tokenizer.save_pretrained("./triage_qwen_lora")

# Optional: Merge LoRA into base model and export for Ollama/Together
# merged = model.merge_and_unload()
# merged.save_pretrained("./triage_qwen_merged")

print("Saved to ./triage_qwen_lora. Download and deploy to Together.ai, Replicate, or Ollama.")

## Deploy Options (Groq does NOT host custom models)

1. **Together.ai** – Upload merged model or LoRA; use their inference API.
2. **Replicate** – Package model as a Replicate model; call via their API.
3. **Ollama** – Convert to GGUF, run locally: `ollama create triage -f Modelfile`.